## 1. Setup

This notebook reproduces the ranking-network comparison used in the thesis.

The ranking engine uses finite-world enumeration. It enumerates the 20 probabilistic fault variables and then propagates the deterministic variables, such as voltages, LEDs, lamp states, and multimeter readings, in topological order. This avoids enumerating all $2^{50}$ complete worlds.

This is a prototype for this circuit. It is not the intended deployed inference method. A general ranking-network tool should use local computation, such as Shenoy-Shafer or join-tree propagation, with rank addition for combination and minimization for marginalization.

The notebook expects these local files in the same repository folder: `ranking_network_V2.py`, `circuit_network.py`, and `bn_network.py`.


In [1]:
import time
import numpy as np
import pandas as pd

from ranking_network_V2 import (
    finite_worlds,
    consistent,
    posterior,
    top_faults,
    explain_posterior,
    INF
)

from circuit_network import (
    build_circuit_network,
    build_circuit_network_fine,
    binary_prior,
    binary_prior_fine,
)

from bn_network import build_bn
from pgmpy.inference import VariableElimination

/Users/marcusgitz/opt/anaconda3/envs/mgi101/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/marcusgitz/opt/anaconda3/envs/mgi101/lib/python3.10/site-packages/pgmpy/estimators/__init__.py:4: FutureWarning: `pgmpy.estimators.StructureScore` is deprecated and will be removed in v1.3.0. Use `pgmpy.structure_score` instead.
  from .StructureScore import (


## 2. Build the ranking networks and the shared BN baseline

The BN and ranking networks use the same variables, parent structure, evidence variables, and local table count. The difference is the local quantification.

The ranking networks are built from `circuit_network.py`:

- `build_circuit_network()` builds the coarse ranking network.
- `build_circuit_network_fine()` builds the fine ranking network.

Deterministic probabilities convert directly:

- probability 1 becomes rank 0.
- probability 0 becomes rank infinity.

The probabilistic entries are converted in two ways:

- coarse: $\kappa = \mathrm{round}(-\log_{10} P)$
- fine: $\kappa = \mathrm{round}(-2\log_{10} P)$

The coarse version gives one rank step per order of magnitude. The fine version gives two rank steps per order of magnitude. This keeps more distance between close probabilities.

The conversion is part of this comparison only. In a deployed ranking-network workflow, ranks should be elicited directly from experts.

The BN is imported as a shared baseline. The real version belongs in the shared BN repository.


In [2]:
def rank_of_fault_state(p_fail, resolution="coarse"):
    """Return the normalized rank assigned to a fault state with probability p_fail."""
    if resolution == "coarse":
        return binary_prior(p_fail, ok="ok", fail="fault")["fault"]
    if resolution == "fine":
        return binary_prior_fine(p_fail, ok="ok", fail="fault")["fault"]
    raise ValueError(f"Unknown resolution: {resolution}")


def parent_sets_match_bn(bn_model, ranking_net):
    """Check that the BN and ranking network use the same nodes and parent sets.

    Parent order may differ between implementations, so this checks sets.
    """
    bn_parent_sets = {
        node: set(bn_model.get_parents(node))
        for node in bn_model.nodes()
    }
    rn_parent_sets = {
        node: set(ranking_net.get_parents(node))
        for node in ranking_net.variables
    }
    return bn_parent_sets == rn_parent_sets

In [3]:
# Build the two ranking networks from the individual circuit-network file.
start = time.time()
rn_coarse = build_circuit_network()
rn_fine = build_circuit_network_fine()
print(f"Ranking networks built in {time.time() - start:.1f}s")
print(rn_coarse)
print(rn_fine)

# Build the shared BN baseline from the baseline file (bn_network.py).
start = time.time()
bn = build_bn(verbose=True)
ve = VariableElimination(bn)
print(f"BN built in {time.time() - start:.1f}s")

# Sanity checks: same variable set and same parent sets across all three models.
assert set(rn_coarse.variables) == set(rn_fine.variables) == set(bn.nodes())
assert parent_sets_match_bn(bn, rn_coarse)
assert parent_sets_match_bn(bn, rn_fine)
print("Variable sets and parent sets match the BN baseline.")

Ranking networks built in 0.0s
RankingNetwork(50 variables, 51 edges)
RankingNetwork(50 variables, 51 edges)
Model valid: True
BN built in 0.0s
Variable sets and parent sets match the BN baseline.


In [4]:
# Enumerate finite worlds once per ranking network.
# Runtime depends on hardware. On the thesis laptop this took about 1 minute per network.
# Later queries reuse these worlds.

start = time.time()
worlds_coarse = list(finite_worlds(rn_coarse))
print(f"coarse: {len(worlds_coarse):,} finite worlds in {time.time() - start:.1f}s")

start = time.time()
worlds_fine = list(finite_worlds(rn_fine))
print(f"fine:   {len(worlds_fine):,} finite worlds in {time.time() - start:.1f}s")

coarse: 1,048,576 finite worlds in 37.7s
fine:   1,048,576 finite worlds in 39.4s


## 3. Small conversion check

The table below shows how the main probabilistic entries map to normalized coarse and fine fault ranks in `circuit_network.py`. It is only a sanity check and not a separate experiment.

In [5]:
conversion_examples = pd.DataFrame([
    {"Quantity": "P(F_PSU_short = yes)", "P(fault)": 0.005},
    {"Quantity": "P(F_battery = exhausted | no PSU short)", "P(fault)": 0.080},
    {"Quantity": "P(F_battery = exhausted | PSU short)", "P(fault)": 0.950},
    {"Quantity": "P(F_sw_N = detached)", "P(fault)": 0.030},
    {"Quantity": "P(F_cable_N = broken)", "P(fault)": 0.020},
    {"Quantity": "P(F_lamp = broken)", "P(fault)": 0.015},
])

conversion_examples["Coarse normalized fault rank"] = conversion_examples["P(fault)"].apply(
    lambda p: rank_of_fault_state(p, "coarse")
)
conversion_examples["Fine normalized fault rank"] = conversion_examples["P(fault)"].apply(
    lambda p: rank_of_fault_state(p, "fine")
)

conversion_examples

,Quantity,P(fault),Coarse normalized fault rank,Fine normalized fault rank
0,P(F_PSU_short = yes),0.005,2,5
1,P(F_battery = exhausted | no PSU short),0.080,1,2
2,P(F_battery = exhausted | PSU short),0.950,0,0
3,P(F_sw_N = detached),0.030,2,3
4,P(F_cable_N = broken),0.020,2,3
5,P(F_lamp = broken),0.015,2,4


## 4. Admissibility and no-evidence sanity check

A ranking network built by adding local tables does not automatically preserve each local table as its marginal. This check compares the no-evidence posterior rank of each fault state with its declared local rank under the normal parent setting.

Most fault variables have no parents. For `F_battery`, the normal parent setting is `F_PSU_short = no`.

This is a sanity check for the constructed network, not a general proof for all ranking networks.


In [6]:
# Fault nodes and their fault states.
# The circuit specification uses the last state as the fault state.

faults = [node for node in rn_coarse.variables if node.startswith("F_")]
fstate = {node: rn_coarse.variables[node][-1] for node in faults}


def declared_fault_rank(net, fault):
    """Return the declared local rank for the fault state.

    For F_battery, use the normal parent setting F_PSU_short=no.
    For all other fault nodes, use the root prior table.
    """
    state = net.variables[fault][-1]
    parent_values = ("no",) if fault == "F_battery" else ()
    return net.kappa_tables[fault][parent_values][state]


def no_evidence_fault_rank(net, fault, worlds):
    """Return the posterior rank of the fault state with no evidence."""
    state = net.variables[fault][-1]
    return posterior(net, query=fault, evidence={}, worlds=worlds)[state]


admissibility_rows = []
for fault in faults:
    coarse_declared = declared_fault_rank(rn_coarse, fault)
    coarse_posterior = no_evidence_fault_rank(rn_coarse, fault, worlds_coarse)
    fine_declared = declared_fault_rank(rn_fine, fault)
    fine_posterior = no_evidence_fault_rank(rn_fine, fault, worlds_fine)

    admissibility_rows.append({
        "Fault": fault,
        "State": fstate[fault],
        "Coarse declared": coarse_declared,
        "Coarse no-evidence posterior": coarse_posterior,
        "Fine declared": fine_declared,
        "Fine no-evidence posterior": fine_posterior,
        "Matches": (
            coarse_declared == coarse_posterior
            and fine_declared == fine_posterior
        ),
    })

admissibility = pd.DataFrame(admissibility_rows)
display(admissibility)

assert admissibility["Matches"].all()
print("Admissibility sanity check passed for all fault nodes.")

,Fault,State,Coarse declared,Coarse no-evidence posterior,Fine declared,Fine no-evidence posterior,Matches
0,F_PSU_short,yes,2,2,5,5,True
1,F_sw_1,detached,2,2,3,3,True
2,F_sw_2,detached,2,2,3,3,True
3,F_sw_3,detached,2,2,3,3,True
4,F_sw_4,detached,2,2,3,3,True
5,F_sw_5,detached,2,2,3,3,True
6,F_sw_6,detached,2,2,3,3,True
7,F_sw_7,detached,2,2,3,3,True
8,F_sw_8,detached,2,2,3,3,True
9,F_cable_1,broken,2,2,3,3,True


Admissibility sanity check passed for all fault nodes.


## 5. Scenario evidence

The evidence below is the transcribed from the scenario descriptions into the variable names used by the BN and rnaking network. It is not pulled automatically from `scenarios.py`. That files defines the fault manipulations in the toy environment. The notebook defines what the diagnostic models observe.

Missing observations are left unobserved. They are not encoded as `off`. This matters in scenario 11, where the control-module indicators are absent. Encoding them as `off` would incorrectly add measurements that were not made.

The thesis uses scenarios 7, 8, 11, and 14. Scenarios 9, 10, and 13 are excluded because they repeat behavior already covered. Scenarios 9 and 10 mirror the module-localization case. Scenario 13 is observationally identical to scenario 7 because the exhausted battery masks the module-3 fault.


In [7]:
S = {}
#scenario 7 - battery exhausted, multimeter available
S[7] = {
    "O_PSU_LED": "off",
    **{f"O_Ind_{n}": "off" for n in range(1,9)},
    "O_Lamp": "off",
    "O_Lamp_indicator": "off",
    "M_battery": "0V",          # measures the battery, shows 0V
    "M_PSU_short": "high",      # no short
}

# scneario 8 - The switch in the control module 3 is detached from one of the corresponding cables, multimeter available
S[8] = {
    "O_PSU_LED": "on",
    "O_Ind_1": "on",
    "O_Ind_2": "on",
    **{f"O_Ind_{n}": "off" for n in range(3,9)},
    "O_Lamp": "off",
    "O_Lamp_indicator": "off",
    "M_battery": "12V",         # measures the battery, shows 12V
    "M_PSU_short": "high",      # no short
}

# scenario 11 - switch 6 detached, all switch indicator LEDs removed,
# NO multimeter.
S[11] = {
    "O_PSU_LED": "on",
    # NO inclusion of the O_IND_n variables, simulating the case where all indicator LEDs are physically removed
    "O_Lamp": "off",
    "O_Lamp_indicator": "off",
    # No inclusion of M_* (no multimeter available)
}

# scenario 14 - PSU hort, battery violently discharged.
# Observartionally identical to scenario 7 EXCEPT for M_PSU_short.
S[14] = {
    "O_PSU_LED": "off",
    **{f"O_Ind_{n}": "off" for n in range(1,9)},
    "O_Lamp": "off",
    "O_Lamp_indicator": "off",
    "M_battery": "0V",          # measures the battery, shows 0V
    "M_PSU_short": "low",       # short detected
}

selected_scenarios = [7, 8, 11, 14]

## 6. Comparison functions

The BN returns posterior probabilities. The ranking networks return posterior ranks. These numbers are not compared as equal quantities. The comparison uses diagnostic orderings, meaning which fault state is most plausible, which hypotheses tie, and whether the same fault region is identified.

In [8]:
def all_fault_ranks(net, evidence, worlds):
    """Compute the posterior rank of every fault state in one pass over worlds."""
    base = INF
    best = {fault: INF for fault in faults}

    for world, kappa in worlds:
        if not consistent(world, evidence):
            continue
        if kappa < base:
            base = kappa
        for fault in faults:
            if world[fault] == fstate[fault] and kappa < best[fault]:
                best[fault] = kappa
        
    if base == INF:
        raise ValueError("evidence impossible under this network")
    
    return {
        fault: (best[fault] - base if best[fault] != INF else INF) 
        for fault in faults
        }

def bn_fault_probs(evidence):
    """Compute the BN posterior probability of each fault state."""
    probs = {}
    for fault in faults:
        query_result = ve.query(
            variables=[fault], 
            evidence=evidence, 
            show_progress=False
        )
    
        probs[fault] = float(query_result.get_value(**{fault: fstate[fault]}))
    return probs
   

def comparison_table(scenario):
    """Return BN probabilities and ranking network ranks for one scenario."""
    evidence = S[scenario]
    coarse_ranks = all_fault_ranks(rn_coarse, evidence, worlds_coarse)
    fine_ranks = all_fault_ranks(rn_fine, evidence, worlds_fine)
    bn_probs = bn_fault_probs(evidence)
    
    df = pd.DataFrame({
        "Fault": faults,
        "BN_Probability": [round(bn_probs[fault], 4) for fault in faults],
        "Coarse_Rank": [coarse_ranks[fault] for fault in faults],
        "Fine_Rank": [fine_ranks[fault] for fault in faults],
    }).sort_values(
        ["BN_Probability", "Coarse_Rank"], 
        ascending=[False, True]
        ).reset_index(drop=True)
    
    return df

def bn_top_faults(df, tol=1e-9):
    max_prob = df["BN_Probability"].max()
    return sorted(df.loc[np.isclose(df["BN_Probability"], max_prob, atol=tol), "Fault"])

def rank0_faults(df, column):
    return sorted(df.loc[df[column] == 0, "Fault"])

def short_fault_list(faults):
    if len(faults) > 6:
        return f"All {len(faults)} faults"
    return ", ".join(faults)

## 7. Per-scenario results

Each scenario prints the ranking network top fault first, followed by the BN and ranking comparison table. The same loop is used for all scenarios. The evidence dictionaries are explicit, but the inference code is not hardcoded per scenario.

In [9]:
def show_ranking_top_faults(scenario, k=8):
    evidence = S[scenario]
    print(f"Scenario {scenario}: ranking network top faults")

    print("Coarse")
    for var, state, rank in top_faults(rn_coarse, evidence, k=k, worlds=worlds_coarse):
        print(f"  {var}={state}, rank={rank}")

    print("Fine")
    for var, state, rank in top_faults(rn_fine, evidence, k=k, worlds=worlds_fine):
        print(f"  {var}={state}, rank={rank}")


def verdict(scenario, df):
    bn_top     = set(bn_top_faults(df))
    coarse_top = set(rank0_faults(df, "Coarse_Rank"))
    fine_top   = set(rank0_faults(df, "Fine_Rank"))
    
    print(f"Scenario {scenario}: "
          f"BN top={sorted(bn_top)} | "
          f"coarse rank0={len(coarse_top)} faults | "
          f"fine rank0={len(fine_top)} faults | "
          f"coars==fine rank-0 set: {coarse_top == fine_top}")
    
tables = {}

for scenario in selected_scenarios:
    print("=" * 80)
    show_ranking_top_faults(scenario)
    print()

    df = comparison_table(scenario)
    tables[scenario] = df
    display(df)

    verdict(scenario, df)
    print()

Scenario 7: ranking network top faults
Coarse
  F_battery=exhausted, rank=0
  F_sw_1=detached, rank=2
  F_sw_2=detached, rank=2
  F_sw_3=detached, rank=2
  F_sw_4=detached, rank=2
  F_sw_5=detached, rank=2
  F_sw_6=detached, rank=2
  F_sw_7=detached, rank=2
Fine
  F_battery=exhausted, rank=0
  F_sw_1=detached, rank=3
  F_sw_2=detached, rank=3
  F_sw_3=detached, rank=3
  F_sw_4=detached, rank=3
  F_sw_5=detached, rank=3
  F_sw_6=detached, rank=3
  F_sw_7=detached, rank=3



,Fault,BN_Probability,Coarse_Rank,Fine_Rank
0,F_battery,1.000,0.0,0.0
1,F_sw_1,0.030,2.0,3.0
2,F_sw_2,0.030,2.0,3.0
3,F_sw_3,0.030,2.0,3.0
4,F_sw_4,0.030,2.0,3.0
5,F_sw_5,0.030,2.0,3.0
6,F_sw_6,0.030,2.0,3.0
7,F_sw_7,0.030,2.0,3.0
8,F_sw_8,0.030,2.0,3.0
9,F_cable_1,0.020,2.0,3.0


Scenario 7: BN top=['F_battery'] | coarse rank0=1 faults | fine rank0=1 faults | coars==fine rank-0 set: True

Scenario 8: ranking network top faults
Coarse
  F_sw_3=detached, rank=0
  F_cable_3=broken, rank=0
  F_sw_4=detached, rank=2
  F_sw_5=detached, rank=2
  F_sw_6=detached, rank=2
  F_sw_7=detached, rank=2
  F_sw_8=detached, rank=2
  F_cable_4=broken, rank=2
Fine
  F_sw_3=detached, rank=0
  F_cable_3=broken, rank=0
  F_sw_4=detached, rank=3
  F_sw_5=detached, rank=3
  F_sw_6=detached, rank=3
  F_sw_7=detached, rank=3
  F_sw_8=detached, rank=3
  F_cable_4=broken, rank=3



,Fault,BN_Probability,Coarse_Rank,Fine_Rank
0,F_sw_3,0.6073,0.0,0.0
1,F_cable_3,0.4049,0.0,0.0
2,F_sw_4,0.0300,2.0,3.0
3,F_sw_5,0.0300,2.0,3.0
4,F_sw_6,0.0300,2.0,3.0
5,F_sw_7,0.0300,2.0,3.0
6,F_sw_8,0.0300,2.0,3.0
7,F_cable_4,0.0200,2.0,3.0
8,F_cable_5,0.0200,2.0,3.0
9,F_cable_6,0.0200,2.0,3.0


Scenario 8: BN top=['F_sw_3'] | coarse rank0=2 faults | fine rank0=2 faults | coars==fine rank-0 set: True

Scenario 11: ranking network top faults
Coarse
  F_sw_1=detached, rank=0
  F_sw_2=detached, rank=0
  F_sw_3=detached, rank=0
  F_sw_4=detached, rank=0
  F_sw_5=detached, rank=0
  F_sw_6=detached, rank=0
  F_sw_7=detached, rank=0
  F_sw_8=detached, rank=0
Fine
  F_sw_1=detached, rank=0
  F_sw_2=detached, rank=0
  F_sw_3=detached, rank=0
  F_sw_4=detached, rank=0
  F_sw_5=detached, rank=0
  F_sw_6=detached, rank=0
  F_sw_7=detached, rank=0
  F_sw_8=detached, rank=0



,Fault,BN_Probability,Coarse_Rank,Fine_Rank
0,F_sw_1,0.0865,0.0,0.0
1,F_sw_2,0.0865,0.0,0.0
2,F_sw_3,0.0865,0.0,0.0
3,F_sw_4,0.0865,0.0,0.0
4,F_sw_5,0.0865,0.0,0.0
5,F_sw_6,0.0865,0.0,0.0
6,F_sw_7,0.0865,0.0,0.0
7,F_sw_8,0.0865,0.0,0.0
8,F_cable_1,0.0577,0.0,0.0
9,F_cable_2,0.0577,0.0,0.0


Scenario 11: BN top=['F_sw_1', 'F_sw_2', 'F_sw_3', 'F_sw_4', 'F_sw_5', 'F_sw_6', 'F_sw_7', 'F_sw_8'] | coarse rank0=17 faults | fine rank0=17 faults | coars==fine rank-0 set: True

Scenario 14: ranking network top faults
Coarse
  F_PSU_short=yes, rank=0
  F_battery=exhausted, rank=0
  F_sw_1=detached, rank=2
  F_sw_2=detached, rank=2
  F_sw_3=detached, rank=2
  F_sw_4=detached, rank=2
  F_sw_5=detached, rank=2
  F_sw_6=detached, rank=2
Fine
  F_PSU_short=yes, rank=0
  F_battery=exhausted, rank=0
  F_sw_1=detached, rank=3
  F_sw_2=detached, rank=3
  F_sw_3=detached, rank=3
  F_sw_4=detached, rank=3
  F_sw_5=detached, rank=3
  F_sw_6=detached, rank=3



,Fault,BN_Probability,Coarse_Rank,Fine_Rank
0,F_PSU_short,1.000,0,0
1,F_battery,1.000,0,0
2,F_sw_1,0.030,2,3
3,F_sw_2,0.030,2,3
4,F_sw_3,0.030,2,3
5,F_sw_4,0.030,2,3
6,F_sw_5,0.030,2,3
7,F_sw_6,0.030,2,3
8,F_sw_7,0.030,2,3
9,F_sw_8,0.030,2,3


Scenario 14: BN top=['F_PSU_short', 'F_battery'] | coarse rank0=2 faults | fine rank0=2 faults | coars==fine rank-0 set: True



## 8. Detailed posterior examples

This section prints readable posterior state ranks for the most important fault variables. It is meant for inspection, not for th emain thesis table.

In [10]:
key_queries = {
    7: ["F_battery", "F_PSU_short"],
    8: ["F_sw_3", "F_cable_3"],
    11: ["F_sw_6", "F_cable_6", "F_lamp"],
    14: ["F_PSU_short", "F_battery"],
}

for scenario in selected_scenarios:
    print("=" * 80)
    print(f"Scenario {scenario}: selected posterior ranks")

    for query in key_queries[scenario]:
        print("Coarse")
        explain_posterior(
            posterior(rn_coarse, query=query, evidence=S    [scenario], worlds=worlds_coarse),
            label=query,
        )

        print("Fine")
        explain_posterior(
            posterior(rn_fine, query=query, evidence=S[scenario], worlds=worlds_fine),
            label=query,
        )
        
        print()

Scenario 7: selected posterior ranks
Coarse
F_battery:
  exhausted: rank 0, (most plausible)
  good: impossible and ruled out
Fine
F_battery:
  exhausted: rank 0, (most plausible)
  good: impossible and ruled out

Coarse
F_PSU_short:
  no: rank 0, (most plausible)
  yes: impossible and ruled out
Fine
F_PSU_short:
  no: rank 0, (most plausible)
  yes: impossible and ruled out

Scenario 8: selected posterior ranks
Coarse
F_sw_3:
  ok: rank 0, (most plausible)
  detached: rank 0, (most plausible)
Fine
F_sw_3:
  ok: rank 0, (most plausible)
  detached: rank 0, (most plausible)

Coarse
F_cable_3:
  ok: rank 0, (most plausible)
  broken: rank 0, (most plausible)
Fine
F_cable_3:
  ok: rank 0, (most plausible)
  broken: rank 0, (most plausible)

Scenario 11: selected posterior ranks
Coarse
F_sw_6:
  ok: rank 0, (most plausible)
  detached: rank 0, (most plausible)
Fine
F_sw_6:
  ok: rank 0, (most plausible)
  detached: rank 0, (most plausible)

Coarse
F_cable_6:
  ok: rank 0, (most plausible)


## 9. Cross-scenario summary

This table is the compact version of the comparison. It mirrors the result use din the thesis: the ranking networks preserve the same broad diagnostic direction as the BN, but they lose precision where close faults collapse into ties.

In [11]:
summary_rows = []

for scenario in selected_scenarios:
    df = tables[scenario]
    bn_top     = set(bn_top_faults(df))
    coarse_top = set(rank0_faults(df, "Coarse_Rank"))
    fine_top   = set(rank0_faults(df, "Fine_Rank"))
    
    summary_rows.append({
        "Scenario": scenario,
        "BN top fault(s)": short_fault_list(bn_top),
        "BN top probability": round(df["BN_Probability"].max(), 4),
        "Coarse rank0 fault(s)": short_fault_list(coarse_top),
        "Fine rank0 fault(s)": short_fault_list(fine_top),
        "Coarse==Fine rank-0 set": coarse_top == fine_top,
    })

summary = pd.DataFrame(summary_rows)
summary

,Scenario,BN top fault(s),BN top probability,Coarse rank0 fault(s),Fine rank0 fault(s),Coarse==Fine rank-0 set
0,7,F_battery,1.0000,F_battery,F_battery,True
1,8,F_sw_3,0.6073,"F_sw_3, F_cable_3","F_sw_3, F_cable_3",True
2,11,All 8 faults,0.0865,All 17 faults,All 17 faults,True
3,14,"F_PSU_short, F_battery",1.0000,"F_PSU_short, F_battery","F_PSU_short, F_battery",True


In [12]:
def compare_coarse_and_fine_below_top(scenario, top_n=8):
    df = tables[scenario].copy()
    df["Coarse - fine"] = df["Coarse_Rank"] - df["Fine_Rank"]
    return df[["Fault", "BN_Probability", "Coarse_Rank", "Fine_Rank", "Coarse - fine"]].head(top_n)

for scenario in selected_scenarios:
    print(f"Scenario {scenario}: coarse/fine differences among the top BN-ranked faults")
    display(compare_coarse_and_fine_below_top(scenario))

Scenario 7: coarse/fine differences among the top BN-ranked faults


,Fault,BN_Probability,Coarse_Rank,Fine_Rank,Coarse - fine
0,F_battery,1.00,0.0,0.0,0.0
1,F_sw_1,0.03,2.0,3.0,-1.0
2,F_sw_2,0.03,2.0,3.0,-1.0
3,F_sw_3,0.03,2.0,3.0,-1.0
4,F_sw_4,0.03,2.0,3.0,-1.0
5,F_sw_5,0.03,2.0,3.0,-1.0
6,F_sw_6,0.03,2.0,3.0,-1.0
7,F_sw_7,0.03,2.0,3.0,-1.0


Scenario 8: coarse/fine differences among the top BN-ranked faults


,Fault,BN_Probability,Coarse_Rank,Fine_Rank,Coarse - fine
0,F_sw_3,0.6073,0.0,0.0,0.0
1,F_cable_3,0.4049,0.0,0.0,0.0
2,F_sw_4,0.0300,2.0,3.0,-1.0
3,F_sw_5,0.0300,2.0,3.0,-1.0
4,F_sw_6,0.0300,2.0,3.0,-1.0
5,F_sw_7,0.0300,2.0,3.0,-1.0
6,F_sw_8,0.0300,2.0,3.0,-1.0
7,F_cable_4,0.0200,2.0,3.0,-1.0


Scenario 11: coarse/fine differences among the top BN-ranked faults


,Fault,BN_Probability,Coarse_Rank,Fine_Rank,Coarse - fine
0,F_sw_1,0.0865,0.0,0.0,0.0
1,F_sw_2,0.0865,0.0,0.0,0.0
2,F_sw_3,0.0865,0.0,0.0,0.0
3,F_sw_4,0.0865,0.0,0.0,0.0
4,F_sw_5,0.0865,0.0,0.0,0.0
5,F_sw_6,0.0865,0.0,0.0,0.0
6,F_sw_7,0.0865,0.0,0.0,0.0
7,F_sw_8,0.0865,0.0,0.0,0.0


Scenario 14: coarse/fine differences among the top BN-ranked faults


,Fault,BN_Probability,Coarse_Rank,Fine_Rank,Coarse - fine
0,F_PSU_short,1.00,0,0,0
1,F_battery,1.00,0,0,0
2,F_sw_1,0.03,2,3,-1
3,F_sw_2,0.03,2,3,-1
4,F_sw_3,0.03,2,3,-1
5,F_sw_4,0.03,2,3,-1
6,F_sw_5,0.03,2,3,-1
7,F_sw_6,0.03,2,3,-1


## 10. Takeaway

The ranking networks recover the same broad fault region as the BN across all four scenarios. Scenario 7 points to the battery. Scenario 8 points to module 3. Scenario 11 points to the series chain but cannot localize a single module because the key observations are missing. Scenario 14 points to the PSU short and the exhausted battery.

The main cost is precisions. The ranking networks tie hypotheses that the BN separates with close porbabilities. This is visible in the switch-versus-cable distinction in scenario 8 and the broader switch/cable tie in scenairo 11.

The resuls is therefore not a speed claim and not a deployment claim. It shows that ordinal local information can presever the main diagnosis in this toy circuit. The next technical step is a general inference enginge, such as Shenoy-Shafer or join-tree propagation, followed by direct expert rank elicitation and testing on a larger vase with ground-thruth faults.